# 🎙 Stimmenkloner – Sprachboard (XTTS v2)

## ⚡ Vor dem Start
**Laufzeit → Laufzeittyp ändern → T4 GPU → Speichern**

Dann alle 4 Schritte der Reihe nach ausführen. Kein Neustart nötig.

---
## Schritt 1 – Installation
Dauert ca. 3–5 Minuten. Nur einmal nötig.

In [ ]:
!pip install -q coqui-tts
!apt-get install -q -y ffmpeg
print('✓ Installation abgeschlossen')

---
## Schritt 2 – Sprachproben hochladen

- **1–10 Dateien** gleichzeitig auswählen (Ctrl+Klick für mehrere)
- Mindestens **10–30 Sekunden** pro Datei, reine Sprache
- Format: MP3 oder WAV, kein Hintergrundgeräusch
- Tipp: Mehr Dateien mit verschiedenen Sätzen = authentischere Stimme

In [ ]:
from google.colab import files

print('Dateien auswählen (Ctrl+Klick für mehrere)...')
uploaded = files.upload()
voice_files = list(uploaded.keys())
print(f'✓ {len(voice_files)} Datei(en) hochgeladen: {", ".join(voice_files)}')

---
## Schritt 3 – Alle 16 Sätze generieren

XTTS v2 Modell wird beim ersten Mal heruntergeladen (~2 GB, ca. 3 Minuten).

Die Sätze kannst du hier beliebig ändern.

In [ ]:
import os
import torch

# Patch: neuere transformers-Versionen haben isin_mps_friendly entfernt (in PyTorch 2.4+ nicht mehr nötig)
import transformers.pytorch_utils
if not hasattr(transformers.pytorch_utils, 'isin_mps_friendly'):
    transformers.pytorch_utils.isin_mps_friendly = lambda elements, test_elements: torch.isin(elements, test_elements)

from TTS.api import TTS

saetze = [
    # Clips 01–08
    'Wer auf Toilette möchte, hebt bitte die Hand.',
    'Heute geht die erste Runde Bier selbstverständlich auf mich.',
    'Der Herr ist mein Hirte und mein Fahrer ist heute Pascal.',
    'Ich erkenne ein gutes Auto daran, wie bequem der Beifahrersitz ist.',
    'Mein Lieblingsauto ist das, in dem mich andere mitnehmen.',
    'Alkoholische Mitarbeit ist heute ausdrücklich erwünscht.',
    'Ich fühle mich wie 2012 im Bierkönig.',
    'Mein Verantwortungsbereich endet ab dem zweiten Bier.',
    # Clips 09–16
    'Mein Führerschein ist wie Gott: Viele glauben daran, gesehen hat ihn noch keiner.',
    'Andere sammeln Kilometer, ich sammle Mitfahrten.',
    'Ich bin der Beweis dafür, dass man auch ohne Führerschein im Leben völlig in die falsche Richtung fahren kann.',
    'Pascal, dafür dass du mich ständig fährst, zahle ich heute für dich und ich liebe dich.',
    'Wenn der Pegel steigt, sinkt das Niveau.',
    'Der Geist ist willig, aber das Bier ist kalt.',
    'Nich lang schnacken, Kopf in Nacken.',
    'Delirium, Delarium - Voll wie ein Aquarium.',
]

os.environ['COQUI_TOS_AGREED'] = '1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Verwende: {device.upper()}')

print('Lade XTTS v2 Modell (~2 GB)...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✓ Modell bereit\n')

print(f'Generiere {len(saetze)} Sätze mit {len(voice_files)} Sprachreferenz(en)...\n')
for i, text in enumerate(saetze, 1):
    wav_path = f'/content/clip_{i:02d}.wav'
    mp3_path = f'/content/clip_{i:02d}.mp3'

    tts.tts_to_file(
        text=text,
        speaker_wav=voice_files,
        language='de',
        file_path=wav_path
    )

    os.system(f'ffmpeg -i {wav_path} -q:a 2 {mp3_path} -y -loglevel quiet')
    os.remove(wav_path)

    preview = text[:60] + '...' if len(text) > 60 else text
    print(f'  ✓ clip_{i:02d}.mp3 → {preview}')

print('\n✓ Alle 16 Clips fertig!')

---
## Schritt 4 – Herunterladen

In [ ]:
!zip -j /content/sprachboard_clips.zip /content/clip_*.mp3

from google.colab import files
files.download('/content/sprachboard_clips.zip')
print('✓ Download gestartet: sprachboard_clips.zip')

---
## Nächste Schritte
1. ZIP entpacken → `clip_01.mp3` bis `clip_16.mp3`
2. Alle Clips auf GitHub in `audio/` hochladen (bestehende ersetzen)
3. Website zeigt automatisch alle 16 Buttons in 4 Kategorien